## models

In [49]:
import quante as qt

# xxx
qt.generate.matrix.heisenberg_matrix(
    L=10, j=(1.,1.,1.), h=0., pauli=True
)

array([[9., 0., 0., ..., 0., 0., 0.],
       [0., 7., 2., ..., 0., 0., 0.],
       [0., 2., 5., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 5., 2., 0.],
       [0., 0., 0., ..., 2., 7., 0.],
       [0., 0., 0., ..., 0., 0., 9.]])

In [50]:
# Dirac Fermion Model

import quante as qt

qt.generate.matrix.syk4_dirac(
    L=10, Nf=5, J=1.
)

array([[ 0.09944073+0.j        ,  0.02945665-0.01991948j,  0.07996349+0.0811933j , ...,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       [ 0.02945665+0.01991948j,  0.09985726+0.j        ,  0.01024142-0.00137561j, ...,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       [ 0.07996349-0.0811933j ,  0.01024142+0.00137561j,  0.19256993+0.j        , ...,  0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        ],
       ...,
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        , ..., -0.06338535+0.j        , -0.05125821+0.00791262j, -0.02137827+0.03744431j],
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        , ..., -0.05125821-0.00791262j, -0.05344752+0.j        ,  0.10606139+0.02246367j],
       [ 0.        +0.j        ,  0.        +0.j        ,  0.        +0.j        , ..., -0.02137827-0.03744431j,  0.10606139-0.02246367j,  0.09045957+0.j        ]])

## 基本使用

定义哈密顿量：

$$
    H = \sum_{i = 1}^{L - 1} (\sigma^{x}_{i}\sigma^{x}_{i + 1} + \sigma^{y}_{i} \sigma^{y}_{i + 1} + \frac{1}{2}  \sigma^{z}_{i} \sigma^{z}_{i + 1})
$$

In [51]:
import quante as qt
op = qt.generate.operas.spin
L = 4
builder = op.builder()
for i in range(L-1):
    builder += 'xx', [i, i+1], 1.
    builder += 'yy', [i, i+1], 1.
    builder += 'zz', [i, i+1], 1.
ham = builder.build()

basis = qt.generate.basis.spin_basis(L=L, Nup=L//2)
hammat = ham.to_matrix(basis=basis)
hammat

array([[ 0.25,  0.5 ,  0.  ,  0.  ,  0.  ,  0.  ],
       [ 0.5 , -0.75,  0.5 ,  0.5 ,  0.  ,  0.  ],
       [ 0.  ,  0.5 , -0.25,  0.  ,  0.5 ,  0.  ],
       [ 0.  ,  0.5 ,  0.  , -0.25,  0.5 ,  0.  ],
       [ 0.  ,  0.  ,  0.5 ,  0.5 , -0.75,  0.5 ],
       [ 0.  ,  0.  ,  0.  ,  0.  ,  0.5 ,  0.25]])

生成基矢

In [52]:
# 生成具有粒子数生活和动量守恒的基矢
basis = qt.generate.basis.spin_basis(L=4, Nup=2, kblock=0)

# 可以查看基矢的个数：
print("空间维数", basis.Ns)

# # 可以获得某个基矢在全空间中的表示
state = basis.to_full_space(0)

# 可以可视化全空间的基矢：
print("第 0 个基矢：")
basisfull = qt.generate.basis.spin_basis(L=4)
basisfull.show_state(basis[0])

空间维数 2
第 0 个基矢：
↑↑↓↓: (0.5+0j)
↑↓↓↑: (0.5+0j)
↓↑↑↓: (0.5+0j)
↓↓↑↑: (0.5+0j)


In [53]:
# 获得哈密顿量在给定基矢下的矩阵：
mat = ham.to_matrix(basis)
mat

array([[ 0.25      +0.j,  1.06066017+0.j],
       [ 0.70710678+0.j, -0.75      +0.j]])

计算基态能

In [54]:
# 对角化
engs, eigstates = qt.linalg.eigh(mat, k=1)  # 获得最低能量的本征态
engs

array([-1.1160254])

计算纠缠

In [ ]:
# 计算纠缠：
entspect = qt.measure.entanglement_spectrum(eigstates[:,0], L//2, basis)  # 纠缠谱
entspect
qt.measure.entropy(entspect)

np.float64(1.1741418371072039)

## 梯子形系统

```
      0   2   4   6
   ---◻---◻---◻---◻---
      |   |   |   |
   ---◻---◻---◻---◻---
      1   3   5   7
```

In [56]:
j1, j2, j3 = 1.0, 2.0, 1.0
import quante as qt
op = qt.generate.operas.spin

L = 5

H_Spart = j1 * op.sum(op.xx(2*i,2*i+2) + op.yy(2*i,2*i+2) + op.zz(2*i,2*i+2) for i in range(L-1))
H_Lpart = op.sum(j2 * (op.xx(2*i+1,2*i+3) + op.yy(2*i+1,2*i+3)) + j1 * op.zz(2*i+1,2*i+3) for i in range(L-1))
H_SLpart = j3 * op.sum(op.xx(2*i,2*i+1) + op.yy(2*i,2*i+1) + op.zz(2*i,2*i+1) for i in range(L))

H = H_Spart + H_Lpart + H_SLpart
H

SpinOper at 0x2238769b6e0, 
|   x     x       coef. |   y     y       coef. |   z     z       coef. |
|-----------------------|-----------------------|-----------------------|
|   0     1       1.000 |   0     1       1.000 |   0     1       1.000 |
|   0     2       1.000 |   0     2       1.000 |   0     2       1.000 |
|   1     3       2.000 |   1     3       2.000 |   1     3       1.000 |
|   2     3       1.000 |   2     3       1.000 |   2     3       1.000 |
|   2     4       1.000 |   2     4       1.000 |   2     4       1.000 |
|   3     5       2.000 |   3     5       2.000 |   3     5       1.000 |
|   4     5       1.000 |   4     5       1.000 |   4     5       1.000 |
|   4     6       1.000 |   4     6       1.000 |   4     6       1.000 |
|   5     7       2.000 |   5     7       2.000 |   5     7       1.000 |
|   6     7       1.000 |   6     7       1.000 |   6     7       1.000 |
|   6     8       1.000 |   6     8       1.000 |   6     8       1.000 |
|   7     

x 方向和 y 方向都是 zzx 相互作用：

In [57]:
j1, j2, j3 = 1.0, 2.0, 1.0

for L in range(4, 10):
    
    H_Spart = j1 * op.sum(op.xx(2*i,2*i+2) + op.yy(2*i,2*i+2) + op.zz(2*i,2*i+2) for i in range(L-1))
    H_Lpart = op.sum(j2 * (op.xx(2*i+1,2*i+3) + op.yy(2*i+1,2*i+3)) + j1 * op.zz(2*i+1,2*i+3) for i in range(L-1))
    H_SLpart = j3 * op.sum(op.xx(2*i,2*i+1) + op.yy(2*i,2*i+1) + op.zz(2*i,2*i+1) for i in range(L))
    
    H = H_Spart + H_Lpart + H_SLpart
    
    basis = qt.generate.basis.spin_basis(L=2*L, Nup=L)
    mat = H.to_matrix(basis, pauli=False, sparse=True)
    gdeng = qt.linalg.eigvalsh(mat, k=1)[0]
    print(f"L={L}, ground state energy={gdeng}")

L=4, ground state energy=-5.149175306097186
L=5, ground state energy=-6.549539316733609
L=6, ground state energy=-7.967781415010542
L=7, ground state energy=-9.379011867020695
L=8, ground state energy=-10.793329596691667
L=9, ground state energy=-12.206480865439996


与 quspin 的转换

In [58]:
# 对比 quspin 和 quante 的效率（需要在安装 quspin 的环境中运行）
import quante as qt
import time


L = 20
ham = qt.generate.operas.spin.heisenberg_operator(L, j=(1, 1, 1))
ham = ham.expandxy(pauli=False)
quspin_basis = qt.generate.basis.quspin_spin_basis(L=L, pauli=0)
lis = ham.quspin_form()

t = time.time()
mat3 = ham.to_matrix(quspin_basis, sparse=True)
print("quspin time: ", time.time()-t)

basis = qt.generate.basis.spin_basis(L=L)
t = time.time()
mat1 = ham.to_matrix(basis, sparse=True)
print("quante time: ", time.time()-t)

print("diff: ",qt.linalg.norm(mat1 - mat3))

quspin time:  1.524693489074707
quante time:  0.4108576774597168
diff:  0.0


生成矩阵的难点在于, 稀疏矩阵的加法, 它无法利用并行加速

automata 之所以更快是因为, 它最小化了大型稀疏矩阵加法的次数

不同平台本征分解的能力

eigvalsh (real)
|  dim\backend   |  numpy  |  torch  |  torch-cuda  |  matlab  |  matlab-gpu  |  matrix ocupied  |  memory needed  |
|:--------------:|:-------:|:-------:|:------------:|:--------:|:------------:|:----------------:|:---------------:|
|  2^14 = 16384  |    ✓    |    ✓    |       ✓      |    ✓     |      ✓       |       2 G        |      4 G        |
|  2^15 = 32768  |    ✓    |    ✓    |       x      |    ✓     |      ✓       |       8 G        |     16 G        |
|  2^16 = 65536  |    x    |    ?    |       x      |    ✓     |      ✓       |      32 G        |     64 G        |

eigh (real)
|  dim\backend   |  numpy  |  torch  |  torch-cuda  |  matlab  |  matlab-gpu  |
|:--------------:|:-------:|:-------:|:------------:|:--------:|:------------:|
|  2^14 = 16384  |    ✓    |    ✓    |       ✓      |    ✓     |      ✓       |
|  2^15 = 32768  |    x    |    ?    |       x      |    ✓     |      ✓       |
|  2^16 = 65536  |    x    |    ?    |       x      |    ✓     |      ✓       |

## SU(2) 工具

下面函数中参数中 `jmblock = (J, m)`

`J` 可取的值，可取 `L/2`, `L2/2-1`, ... `0`，`m` 可取的值为 `5`, `4`, `3`, `2`, `1`, `0`, `-1`, `-2`, `-3`, `-4`, `-5`

`dim` 表示子空间的维数，`num` 表示子空间重复的次数，因而：

In [59]:
import quante as qt
L = 4
basis = qt.generate.basis.spin_basis(L, jmblock=(2, 2))

basis.print_dims(L)

   J  |   num  |   dim   
-----------------------
  2.0 |   5    |  1
  1.0 |   3    |  3
  0.0 |   1    |  2
-----------------------
note: \sum num * dim = 2^L


可以与普通的 basis 一样生成矩阵，但目前采用投影矩阵的方法，效率低

In [60]:
ham = qt.generate.operas.spin.heisenberg_operator(L)
mat = ham.to_matrix(basis)
mat.shape, mat

((1, 1), array([[0.75]]))

In [61]:
basis_ = qt.generate.basis.spin_basis(L)
mat_ = ham.to_matrix(basis_)
qt.linalg.eigvalsh(mat_).reshape(4,-1)

array([[-1.6160254 , -0.95710678, -0.95710678, -0.95710678],
       [-0.25      , -0.25      , -0.25      ,  0.1160254 ],
       [ 0.45710678,  0.45710678,  0.45710678,  0.75      ],
       [ 0.75      ,  0.75      ,  0.75      ,  0.75      ]])

可以看到 0.75 确实重复的 5 次

In [62]:
basis = qt.generate.basis.spin_basis(L, jmblock=(1, 1))
mat = ham.to_matrix(basis)
qt.linalg.eigvalsh(mat)

array([-0.95710678, -0.25      ,  0.45710678])

对比可以看到 这三个数每个都重复了三次

验证每个基矢都是 $J^2$ 的本征态

In [63]:
vec = basis.to_full_space(1)  # 第二个基矢，任何一个基矢都满足
# 这个向量是 J^2 的本征态

op = qt.generate.operas
op_Jx = op.sum(op.x(i) for i in range(L))
op_Jy = op.sum(op.y(i) for i in range(L))
op_Jz = op.sum(op.z(i) for i in range(L))
op_J2 = op_Jx**2 + op_Jy**2 + op_Jz**2

basis_ = qt.generate.basis.spin_basis(L)
mat_J2 = op_J2.to_matrix(basis_)

import numpy as np
np.real_if_close(mat_J2 @ vec - 1*(1+1)*vec)  # 这个向量是 J^2 的本征态

array([ 0.,  0., -0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.])

验证每个基矢都是 $J_z$ 的本征态

In [64]:
basis_ = qt.generate.basis.spin_basis(L)
mat_Jz = op_Jz.to_matrix(basis_)

import numpy as np
np.real_if_close(mat_Jz @ vec - 1*vec)  # 这个向量是 Jz 的本征态

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

## 算符代数的一些工具

目前主要是费米子的算符代数，后续会添加玻色子算符代数的内容。

In [65]:
import quante as qt
op = qt.generate.operas.fermion

L = 4
builder = op.builder()
for l in range(L):
    builder += '+-', [l, l+1], 1.
    builder += '+-', [l+1, l], 1.
ham = builder.build()
ham

FermionOper at 0x223876bddf0, 
|   +     -       coef. |
|-----------------------|
|   1     0       1.000 |
|   0     1       1.000 |
|   2     1       1.000 |
|   1     2       1.000 |
|   3     2       1.000 |
|   2     3       1.000 |
|   4     3       1.000 |
|   3     4       1.000 |

In [66]:
import quante as qt
op = qt.generate.operas.fermion

builder = op.builder()
builder += '-'*6 + '+'*6, list(range(1, 7))+list(range(1, 7)), 1.
ham = builder.build()
ham


FermionOper at 0x223e37f65a0, 
|   -     -     -     -     -     -     +     +     +     +     +     +       coef. |
|-----------------------------------------------------------------------------------|
|   1     2     3     4     5     6     1     2     3     4     5     6       1.000 |

In [67]:
ham.normal_ordering()

FermionOper at 0x223e3676c00, 
|   I       coef. |   +     -       coef. |   +     +     -     -       coef. |
|-----------------|-----------------------|-----------------------------------|
|   0      -1.000 |   1     1       1.000 |   1     2     1     2       1.000 |
|                 |   2     2       1.000 |   1     3     1     3       1.000 |
|                 |   3     3       1.000 |   2     3     2     3       1.000 |
|                 |   4     4       1.000 |   1     4     1     4       1.000 |
|                 |   5     5       1.000 |   2     4     2     4       1.000 |
|                 |   6     6       1.000 |   3     4     3     4       1.000 |
|                                         |   1     5     1     5       1.000 |
|                                         |   2     5     2     5       1.000 |
|                                         |   3     5     3     5       1.000 |
|                                         |   4     5     4     5       1.000 |
|        

## JW transformation

In [68]:
# JW transformation  spin -> fermion
import quante as qt

op = qt.generate.operas.fermion
builder = op.builder()
L = 10
J, γ = 1, 0.0
for i in range(L-1):
    builder += "+-", [i+1, i], (J+γ)/2
    builder += "-+", [i+1, i], -(J-γ)/2
ham = builder.build()
ham.normal_ordering()
# ham.jw_transfer()

FermionOper at 0x22385decc80, 
|   +     -       coef. |
|-----------------------|
|   1     0       0.500 |
|   0     1       0.500 |
|   2     1       0.500 |
|   1     2       0.500 |
|   3     2       0.500 |
|   2     3       0.500 |
|   4     3       0.500 |
|   3     4       0.500 |
|   5     4       0.500 |
|   4     5       0.500 |
|   6     5       0.500 |
|   5     6       0.500 |
|   7     6       0.500 |
|   6     7       0.500 |
|   8     7       0.500 |
|   7     8       0.500 |
|   9     8       0.500 |
|   8     9       0.500 |

In [69]:
# JW transformation  fermion -> spin

import quante as qt

op = qt.generate.operas.fermion
builder = op.builder()
L = 10
J, γ = 1, 0.0
for i in range(L-1):
    builder += "+-", [i+1, i], (J+γ)/2
    builder += "+-", [i, i+1], (J-γ)/2
ham = builder.build()
ham

FermionOper at 0x22387699eb0, 
|   +     -       coef. |
|-----------------------|
|   1     0       0.500 |
|   0     1       0.500 |
|   2     1       0.500 |
|   1     2       0.500 |
|   3     2       0.500 |
|   2     3       0.500 |
|   4     3       0.500 |
|   3     4       0.500 |
|   5     4       0.500 |
|   4     5       0.500 |
|   6     5       0.500 |
|   5     6       0.500 |
|   7     6       0.500 |
|   6     7       0.500 |
|   8     7       0.500 |
|   7     8       0.500 |
|   9     8       0.500 |
|   8     9       0.500 |

In [70]:
# 验证正确性：
import quante as qt
import numpy as np

op = qt.generate.operas.spin

L = 5
basis = qt.generate.basis.spin_basis(L=L)

ham = op.heisenberg_operator(L=L).expandxy(pauli=True)

mat1 = ham.to_matrix(basis)

ham = ham.jw_transfer()  # spin -> fermion
ham = ham.jw_transfer()  # fermion -> spin

mat2 = ham.to_matrix(basis)
print(np.linalg.norm(mat1 - mat2))

0.0


In [71]:
# automata - fermion

# 验证正确性：
import quante as qt
import numpy as np

op = qt.generate.operas.spin

L = 5
basis = qt.generate.basis.spin_basis(L=L)
ham = op.heisenberg_operator(L=L).expandxy(pauli=True)
mat1 = ham.to_matrix(basis)
ham = ham.jw_transfer()  # spin -> fermion
ham = ham.jw_transfer()  # fermion -> spin
mpo = ham.to_mpo()
mat2 = mpo.to_matrix().numpy()
print(np.linalg.norm(mat1 - mat2))
print(mpo)

0.0
MPO;  torch.float64;  norm: 1.960e+01;  maxbonddim: 5;  device: cpu;
physdim:    2|    2|    2|    2|    2| 
         ----O-----O-----O-----O-----O----
physdim:    2|    2|    2|    2|    2| 
bonddim:  1     5     5     5     5     1
site:        0     1     2     3     4  


## Spinfull Femion

In [72]:
op = qt.generate.operas.spinful_fermion
builder = op.builder()
builder += '+-|', [0, 1], 1.
builder += '|+-', [1, 0], 1.
builder += '+|-', [1, 0], 1.

In [73]:
import quante as qt
op = qt.generate.operas.spinful_fermion

ham = op.Fermi_Hubbard_operator(L=5)
print(ham)


SpinfulFermionOper (SpinUp | SpinDown) at 0x223e3676cc0, 
|   +     -   |   coef. |   -     +   |   coef. | | +     -       coef. |
|-----------------------|-----------------------|-----------------------|
|   0     1      -1.000 |   0     1       1.000 |   0     1      -1.000 |
|   1     2      -1.000 |   1     2       1.000 |   1     2      -1.000 |
|   2     3      -1.000 |   2     3       1.000 |   2     3      -1.000 |
|   3     4      -1.000 |   3     4       1.000 |   3     4      -1.000 |
| | -     +       coef. |   n   | n       coef. |
|-----------------------|-----------------------|
|   0     1       1.000 |   0     0       5.000 |
|   1     2       1.000 |   1     1       5.000 |
|   2     3       1.000 |   2     2       5.000 |
|   3     4       1.000 |   3     3       5.000 |
|                       |   4     4       5.000 |



In [74]:
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham


SpinfulFermionOper (SpinUp | SpinDown) at 0x22385cd6cc0, 
|   +     -   |   coef. |   -     +   |   coef. | | +     -       coef. |
|-----------------------|-----------------------|-----------------------|
|   0     1      -1.000 |   0     1       1.000 |   0     1      -1.000 |
|   1     2      -1.000 |   1     2       1.000 |   1     2      -1.000 |
|   2     3      -1.000 |   2     3       1.000 |   2     3      -1.000 |
|   3     4      -1.000 |   3     4       1.000 |   3     4      -1.000 |
| | -     +       coef. |   n   | n       coef. |
|-----------------------|-----------------------|
|   0     1       1.000 |   0     0       5.000 |
|   1     2       1.000 |   1     1       5.000 |
|   2     3       1.000 |   2     2       5.000 |
|   3     4       1.000 |   3     3       5.000 |
|                       |   4     4       5.000 |

In [75]:
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()

basis = qt.generate.basis.quspin_spinful_fermion_basis(L=L)
mat = ham.to_matrix(basis, dtype=np.float64)
mat

array([[25.,  0.,  0., ...,  0.,  0.,  0.],
       [ 0., 20., -1., ...,  0.,  0.,  0.],
       [ 0., -1., 20., ...,  0.,  0.,  0.],
       ...,
       [ 0.,  0.,  0., ...,  0., -1.,  0.],
       [ 0.,  0.,  0., ..., -1.,  0.,  0.],
       [ 0.,  0.,  0., ...,  0.,  0.,  0.]])

In [76]:
# 展成 spinless fermion
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham = ham.to_spinless(mode='extend')
ham

FermionOper at 0x223e36e1f40, 
|   +     -       coef. |   -     +       coef. |   n     n       coef. |
|-----------------------|-----------------------|-----------------------|
|   0     1      -1.000 |   0     1       1.000 |   0     5       5.000 |
|   1     2      -1.000 |   1     2       1.000 |   1     6       5.000 |
|   2     3      -1.000 |   2     3       1.000 |   2     7       5.000 |
|   3     4      -1.000 |   3     4       1.000 |   3     8       5.000 |
|   5     6      -1.000 |   5     6       1.000 |   4     9       5.000 |
|   6     7      -1.000 |   6     7       1.000 |
|   7     8      -1.000 |   7     8       1.000 |
|   8     9      -1.000 |   8     9       1.000 |

In [77]:
# 展成 spinless fermion
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham = ham.to_spinless(mode='extend')

basis = qt.generate.basis.quspin_fermion_basis(L=2*L)
mat = ham.to_matrix(basis, dtype=np.float64)
mat, np.linalg.eigvalsh(mat)[0]

(array([[25.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0., 20., -1., ...,  0.,  0.,  0.],
        [ 0., -1., 20., ...,  0.,  0.,  0.],
        ...,
        [ 0.,  0.,  0., ...,  0., -1.,  0.],
        [ 0.,  0.,  0., ..., -1.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]]),
 np.float64(-3.382617960996765))

In [78]:
# 展成 spinless fermion
import quante as qt
op = qt.generate.operas.spinful_fermion

L = 5
builder = op.builder()
for i in range(L-1):
    builder += "+-|", [i, i+1], -1.0
    builder += "-+|", [i, i+1], 1.0
    builder += "|+-", [i, i+1], -1.0
    builder += "|-+", [i, i+1], 1.0
for i in range(L):
    builder += "n|n", [i, i], 5.0
ham = builder.build()
ham = ham.to_spinless(mode='near')

basis = qt.generate.basis.quspin_fermion_basis(L=2*L)
mat = ham.to_matrix(basis, dtype=np.float64)
mat, np.linalg.eigvalsh(mat)[0]

(array([[25.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0., 20.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0., 20., ...,  0.,  0.,  0.],
        ...,
        [ 0.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.],
        [ 0.,  0.,  0., ...,  0.,  0.,  0.]]),
 np.float64(-3.382617960996697))